# OCR / image→text benchmark — Colab thin shell

All logic in [`scripts/`](https://github.com/sondreskarsten/norwegian-ocr-benchmark/tree/main/scripts). This notebook only orchestrates: auth → install → download PDFs → launch parallel engine processes → live monitor.

**Architecture:**

- **Calibration** (one-off, run on a spot VM, NOT in this notebook):   measures per-engine init-time, p50/p95 wall, VRAM/RAM peak, computes a parallelism plan,   writes `gs://sondre_brreg_data/raw/ocr_bench_11k/_calibration.json`.
- **Sweep** (this notebook): launches each engine as an independent background process   (`subprocess.Popen`), each writing per-PDF result blobs to GCS and a JSONL heartbeat   to `_heartbeat/{engine}.jsonl`.
- **Resume**: every script skips orgnrs that already have a result blob. Crash, preempt,   or close the tab — nothing is lost.

---

## 1 · Auth (Colab built-in)

In [2]:
from google.colab import auth as colab_auth
colab_auth.authenticate_user()

PROJECT = 'sondreskarsten-d7d14'
import os
os.environ['GOOGLE_CLOUD_PROJECT'] = PROJECT

!gcloud config set project {PROJECT} 2>&1 | tail -1

from google.cloud import storage
cli = storage.Client(project=PROJECT)
print('project:', cli.project)
print('bucket sondre_brreg_data exists:', cli.bucket('sondre_brreg_data').exists())

MessageError: Error: credential propagation was unsuccessful

## 2 · Clone repo + install base deps

In [ ]:
!cd /content && rm -rf norwegian-ocr-benchmark
!cd /content && git clone -q https://github.com/sondreskarsten/norwegian-ocr-benchmark.git
%cd /content/norwegian-ocr-benchmark
!pip install -q -r requirements.txt 2>&1 | tail -3

import sys
sys.path.insert(0, '/content/norwegian-ocr-benchmark')

import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
    print('VRAM total MiB:', torch.cuda.get_device_properties(0).total_memory // (1024*1024))

## 3 · Pull all PDFs locally with rclone

Engines never touch GCS during inference — they read from `/content/pdfs/{orgnr}/aarsregnskap_{year}.pdf`.

In [ ]:
!python -m scripts.download_pdfs

## 4 · Pull the latest calibration plan

Run `python -m scripts.calibrate --n 10` on a spot VM first if no calibration exists yet.

In [ ]:
from google.cloud import storage
import json

cli = storage.Client()
blob = cli.bucket('sondre_brreg_data').blob('raw/ocr_bench_11k/_calibration.json')
if not blob.exists():
    raise SystemExit('NO CALIBRATION FOUND. Run `python -m scripts.calibrate --n 10` on a spot VM first.')
cal = json.loads(blob.download_as_text())
open('engine_calibration.json','w').write(json.dumps(cal, indent=2, default=str))
print('plan:')
print(json.dumps(cal.get('plan'), indent=2))

## 5 · Launch parallel engine processes

Each engine runs as an independent background subprocess. CPU engines all run truly parallel; GPU engines are bucketed into groups whose summed VRAM fits the device. This cell returns once everything is launched — actual sweep continues in the background.

Edit `MAX_PDFS` to limit per engine. Set to `''` for full 12,879 sweep.

In [ ]:
import subprocess, sys

MAX_PDFS = ''  # '' = full sweep; '500' = first 500 PDFs per engine

cmd = [sys.executable, '-m', 'scripts.parallel_launcher',
       '--calibration', 'engine_calibration.json']
if MAX_PDFS:
    cmd += ['--max-pdfs', str(MAX_PDFS)]

print('starting:', ' '.join(cmd))
p = subprocess.Popen(cmd, stdout=sys.stdout, stderr=sys.stdout)
print('parent launcher pid:', p.pid)
print('individual engine logs are at /content/norwegian-ocr-benchmark/logs/{engine}.log')

## 6 · Live monitor — tails heartbeats from GCS

Refreshes every 30s. Safe to interrupt; engines keep running. Re-run anytime.

In [ ]:
!python -m scripts.monitor --watch

## 7 · Aggregate — once the sweep is done

In [ ]:
!python -m scripts.aggregate